# الفصل 21: التوازي الموزّع للتدريب

> طبقة الحدس لنأخذ مثالاً ملموسًا: نموذج بحجم 70 مليار معامل (مثل LLaMA-2 70B).

- **الجزء:** 5 — بناء LLM من الصفر
- **الفصل:** 21
- **المستوى:** خبير
- **زمن القراءة:** 30 دقيقة
- **المتطلبات:** الفصل 20: قوانين التوسّع: Chinchilla
- **الوسوم:** Data Parallelism، Tensor Parallelism، Pipeline Parallelism، ZeRO، FSDP

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch21-distributed-training.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-05-building-llms%2Fch21-distributed-training.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-05-building-llms/ch21-distributed-training.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch21-distributed-training.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch21-distributed-training.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch21-distributed-training.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch21-distributed-training.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-05-building-llms%2Fch21-distributed-training.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-05-building-llms/ch21-distributed-training.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch21-distributed-training.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch21-distributed-training.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch21-distributed-training.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> طبقة الحدس لنأخذ مثالاً ملموسًا: نموذج بحجم 70 مليار معامل (مثل LLaMA-2 70B).

> **بيانات الفصل**
>
> - **الجزء:** 5 — بناء LLM من الصفر
> - **المستوى:** خبير
> - **زمن القراءة:** 30 دقيقة
> - **المتطلبات:** الفصل 20: قوانين التوسّع: Chinchilla


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Data Parallelism (DP) — الأساس
- فهم Tensor Parallelism (TP)
- فهم Pipeline Parallelism (PP)


## 1. لماذا يهمّك هذا الموضوع؟

> 🟢 **طبقة الحدس**

لنأخذ مثالاً ملموسًا: نموذج بحجم 70 مليار معامل (مثل LLaMA-2 70B). لو خزّناه بدقة FP32 (4 بايت لكل معامل)، فقط الأوزان تحتاج:

$$70 \times 10^9 \times 4 \text{ bytes} = 280 \text{ GB}$$

وهذا **فقط للأوزان**. أثناء التدريب نحتاج أيضًا:
- **التدرجات (Gradients)**: نفس حجم الأوزان تقريبًا (280 GB إضافية)
- **حالات المُحسِّن (Optimizer States)**: AdamW يحتفظ بمتوسطين متحركين (m وv) لكل معامل، أي ضعف حجم الأوزان (560 GB إضافية بدقة FP32)
- **التنشيطات الوسيطة (Activations)** المطلوبة لـ Backpropagation، والتي تعتمد على حجم الدفعة (Batch Size) وطول التسلسل

المجموع التقريبي لأوزان+تدرجات+حالات محسِّن فقط: **~1120 GB**! بينما أقوى GPU تدريب شائع (NVIDIA H100) يملك فقط **80 GB** من ذاكرة HBM. هذا يعني أننا نحتاج **ما لا يقل عن 14 وحدة GPU فقط لتخزين الحالة الأساسية**، ناهيك عن التنشيطات والمساحة اللازمة فعليًا للحوسبة.

هذا هو جوهر مشكلة **التدريب الموزَّع (Distributed Training)**: كيف نُوزِّع نموذجًا وبياناته وحساباته عبر مئات أو آلاف الـ GPUs بكفاءة، مع تقليل زمن الاتصال الشبكي بينها (الذي يصبح غالبًا عنق الزجاجة الحقيقي، وليس الحوسبة نفسها)؟


## 2. الخريطة الذهنية


```text
استراتيجيات التوازي الموزَّع
│
├── 1. Data Parallelism (DP) — توازي البيانات
│     كل GPU يملك نسخة كاملة من النموذج، يعالج دفعة بيانات مختلفة
│     المشكلة: لا يحل مشكلة عدم اتساع النموذج في ذاكرة GPU واحد
│
├── 2. Tensor Parallelism (TP) — توازي المُوتِرات (داخل الطبقة)
│     تقسيم كل مصفوفة وزن (Weight Matrix) نفسها عبر عدة GPUs
│     يحتاج اتصالًا شبكيًا سريعًا جدًا (NVLink) لأنه يحدث داخل كل طبقة
│
├── 3. Pipeline Parallelism (PP) — التوازي عبر المراحل
│     تقسيم طبقات النموذج (مثلاً: طبقات 1-10 على GPU1، 11-20 على GPU2)
│     المشكلة: "فقاعات" توقف (Bubbles) أثناء انتظار المراحل بعضها بعضًا
│
├── 4. ZeRO (Zero Redundancy Optimizer) — تحسين توازي البيانات
│     تقسيم حالات المُحسِّن/التدرجات/الأوزان بدلاً من تكرارها في كل GPU
│     ZeRO-1 (تقسيم Optimizer States)
│     ZeRO-2 (+ تقسيم Gradients)
│     ZeRO-3 (+ تقسيم Model Parameters نفسها)
│
├── 5. FSDP (Fully Sharded Data Parallel) — تطبيق PyTorch الأصلي لمبدأ ZeRO-3
│
└── 6. التوازي الهجين (3D Parallelism)
      الجمع بين DP + TP + PP معًا لتدريب النماذج الأضخم (مئات المليارات)
```


## 3. الشرح المتدرج

### 3.1 Data Parallelism (DP) — الأساس

> 🟡 **طبقة الآلية**

أبسط أشكال التوازي: كل GPU يملك **نسخة كاملة متطابقة** من النموذج. البيانات الكلية (Global Batch) تُقسَّم إلى دفعات فرعية (Micro-batches)، كل GPU يعالج دفعته الفرعية بشكل مستقل (Forward + Backward)، ثم تُجمَّع (تُزامَن) التدرجات بين كل الـ GPUs عبر عملية **All-Reduce** (جمع ثم بث المتوسط لجميع العقد)، وأخيرًا يُحدِّث كل GPU نسخته من الأوزان بنفس التدرج المُجمَّع — مما يضمن بقاء كل النسخ متطابقة بعد كل خطوة.


In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def setup_distributed_data_parallel(rank: int, world_size: int):
    """
    إعداد بيئة PyTorch Distributed للتدريب بتوازي البيانات.
    'rank' هو رقم تعريف العملية الحالية (0, 1, 2, ...)،
    'world_size' هو إجمالي عدد العمليات (عادة = عدد GPUs).
    """
    dist.init_process_group(
        backend="nccl",  # NCCL هو backend المُحسَّن للاتصال بين GPUs من NVIDIA
        rank=rank,
        world_size=world_size,
    )
    torch.cuda.set_device(rank)

def train_step_with_data_parallelism(model: DDP, optimizer,
                                       micro_batch_input, micro_batch_labels):
    """
    خطوة تدريب واحدة بتوازي البيانات. الأهم هنا أن استدعاء loss.backward()
    يُفعِّل تلقائيًا (ضمنيًا عبر DDP hooks) عملية All-Reduce لتدرجات كل
    المعاملات عبر كل الـ GPUs المشاركة، بحيث يحصل كل GPU على متوسط
    التدرجات من كل النسخ قبل تحديث الأوزان محليًا.
    """
    optimizer.zero_grad()

    logits = model(micro_batch_input)
    loss = nn.functional.cross_entropy(
        logits.view(-1, logits.size(-1)), micro_batch_labels.view(-1)
    )

    # هذا الاستدعاء يُشغِّل All-Reduce تلقائيًا بفضل DDP
    loss.backward()

    # كل GPU يُحدِّث أوزانه محليًا، لكن بنفس التدرج المُتوسَّط بالضبط
    # مما يضمن تطابق كل النسخ بعد كل خطوة
    optimizer.step()

    return loss.item()

def demonstrate_allreduce_communication_cost(num_params: float,
                                               num_gpus: int,
                                               bandwidth_gbps: float = 400):
    """
    تقدير زمن الاتصال اللازم لعملية All-Reduce لتدرجات نموذج بحجم
    num_params معامل، عبر num_gpus وحدة GPU متصلة بسرعة bandwidth_gbps
    (NVLink الحديث يوفر ~400-900 GB/s بين GPUs داخل نفس العقدة).

    نستخدم تقريب خوارزمية Ring-AllReduce، التي تنقل بيانات بحجم
    يقارب 2×(num_gpus-1)/num_gpus من حجم التدرجات الكلي عبر الشبكة،
    بغض النظر عمليًا عن عدد الـ GPUs (خاصية مهمة لقابلية التوسع).
    """
    gradient_size_bytes = num_params * 4  # افتراض FP32 للتدرجات
    gradient_size_gb = gradient_size_bytes / 1e9

    # خوارزمية Ring All-Reduce: الحجم المنقول تقريبًا ثابت تقريبيًا
    # بغض النظر عن num_gpus (لا يتوسع خطيًا سلبًا مع عدد العقد)
    data_transferred_gb = 2 * gradient_size_gb * (num_gpus - 1) / num_gpus

    communication_time_sec = data_transferred_gb / bandwidth_gbps

    print(f"حجم النموذج: {num_params/1e9:.1f} مليار معامل")
    print(f"حجم التدرجات (FP32): {gradient_size_gb:.2f} GB")
    print(f"عدد GPUs: {num_gpus}")
    print(f"زمن All-Reduce التقديري: {communication_time_sec*1000:.2f} ميلي ثانية")

demonstrate_allreduce_communication_cost(num_params=7e9, num_gpus=8)


> ⚠️ **خطأ شائع**: الاعتقاد بأن Data Parallelism "يحل مشكلة النماذج الضخمة". في الحقيقة DP **لا يقسّم النموذج نفسه إطلاقًا** — كل GPU يحتاج ذاكرة كافية لاستيعاب نسخة كاملة من النموذج وحالات مُحسِّنه. DP يحل فقط مشكلة **سرعة معالجة كمية كبيرة من البيانات بالتوازي**، وليس مشكلة "النموذج لا يتسع في GPU واحد".

### 3.2 Tensor Parallelism (TP)

> 🔴 **طبقة التعمّق**

عندما لا يتسع **النموذج نفسه** في ذاكرة GPU واحد، نحتاج تقسيم الأوزان ذاتها. Tensor Parallelism يقسّم **مصفوفة وزن واحدة** عبر عدة GPUs. المثال الكلاسيكي: طبقة Feed-Forward في Transformer، التي تتضمن ضرب مصفوفة بحجم $(d_{model}, d_{ff})$:

$$Y = XW_1, \quad W_1 \in \mathbb{R}^{d_{model} \times d_{ff}}$$

يمكننا تقسيم $W_1$ **عموديًا (Column-wise)** عبر N من الـ GPUs، بحيث يملك كل GPU شريحة $(d_{model}, d_{ff}/N)$ فقط:

$$W_1 = [W_1^{(1)}, W_1^{(2)}, \ldots, W_1^{(N)}]$$

كل GPU يحسب $Y^{(i)} = X W_1^{(i)}$ بشكل مستقل تمامًا (لا حاجة لاتصال شبكي في هذه الخطوة!)، لأن كل GPU يحتاج فقط X الكاملة (تُبَث له) ليحسب شريحته من الناتج.

ثم في الطبقة الثانية من FFN ($W_2 \in \mathbb{R}^{d_{ff} \times d_{model}}$)، نقسّم $W_2$ **أفقيًا (Row-wise)** بحيث تتطابق الأبعاد:

$$Z = Y W_2 = \sum_{i=1}^{N} Y^{(i)} W_2^{(i)}$$

هذا المجموع يتطلب عملية **All-Reduce واحدة فقط** في نهاية كل طبقة FFN، وهذا هو جوهر تصميم **Megatron-LM** (نهج NVIDIA الشهير لتوازي المُوتِرات).


In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist

class ColumnParallelLinear(nn.Module):
    """
    طبقة خطية مُقسَّمة عموديًا عبر عدة GPUs (Megatron-LM style).
    كل GPU يملك عمودًا فرعيًا من مصفوفة الوزن الكاملة، ويحسب شريحته
    من الناتج بشكل مستقل تمامًا دون أي اتصال شبكي أثناء الحساب نفسه.
    """
    def __init__(self, in_features: int, out_features: int,
                 world_size: int, rank: int):
        super().__init__()
        assert out_features % world_size == 0, \
            "عدد المخرجات يجب أن يقبل القسمة على عدد GPUs"
        self.out_features_per_partition = out_features // world_size
        self.rank = rank
        self.world_size = world_size

        # كل GPU يُنشئ فقط شريحته الخاصة من مصفوفة الوزن الكاملة
        self.weight = nn.Parameter(
            torch.randn(in_features, self.out_features_per_partition) * 0.02
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x هنا كاملة (مُكرَّرة/مبثوثة على كل GPU)، والناتج شريحة فقط
        return x @ self.weight  # [batch, seq, out_features_per_partition]

class RowParallelLinear(nn.Module):
    """
    طبقة خطية مُقسَّمة أفقيًا. المدخل هنا هو شريحة (ناتج من طبقة
    ColumnParallel سابقة)، والناتج يتطلب عملية All-Reduce لجمع
    المساهمات من كل الـ GPUs قبل الحصول على الناتج الكامل الصحيح.
    """
    def __init__(self, in_features: int, out_features: int,
                 world_size: int, rank: int, process_group=None):
        super().__init__()
        assert in_features % world_size == 0
        self.in_features_per_partition = in_features // world_size
        self.rank = rank
        self.world_size = world_size
        self.process_group = process_group

        self.weight = nn.Parameter(
            torch.randn(self.in_features_per_partition, out_features) * 0.02
        )

    def forward(self, x_partition: torch.Tensor) -> torch.Tensor:
        partial_output = x_partition @ self.weight

        # All-Reduce: جمع المساهمات الجزئية من كل الـ GPUs
        # هذه هي نقطة الاتصال الشبكي الوحيدة في مسار TP لهذه الطبقة
        if dist.is_initialized():
            dist.all_reduce(partial_output, op=dist.ReduceOp.SUM,
                             group=self.process_group)
        return partial_output

class TensorParallelFFN(nn.Module):
    """
    طبقة Feed-Forward كاملة مُقسَّمة بتوازي المُوتِرات بنمط Megatron-LM:
    - الطبقة الأولى: Column-Parallel (بدون اتصال شبكي)
    - التفعيل: يُطبَّق محليًا على كل شريحة
    - الطبقة الثانية: Row-Parallel (اتصال All-Reduce واحد فقط في النهاية)

    النتيجة: اتصال شبكي واحد فقط لكل طبقة FFN كاملة، بدلاً من
    اتصال بعد كل عملية ضرب مصفوفات منفردة.
    """
    def __init__(self, d_model: int, d_ff: int, world_size: int, rank: int):
        super().__init__()
        self.fc1 = ColumnParallelLinear(d_model, d_ff, world_size, rank)
        self.fc2 = RowParallelLinear(d_ff, d_model, world_size, rank)
        self.activation = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = self.activation(self.fc1(x))  # بدون اتصال شبكي
        output = self.fc2(hidden)               # اتصال All-Reduce واحد هنا
        return output


> 💡 **نصيحة عملية**: Tensor Parallelism يتطلب اتصالاً شبكيًا **متكررًا جدًا وسريعًا جدًا** (مرة لكل طبقة تقريبًا) لأن البيانات تُنقَل أثناء الحساب نفسه داخل كل طبقة. لهذا السبب يُستخدَم تقريبًا **حصرًا داخل نفس العقدة الفيزيائية الواحدة** (Single Node) حيث تتوفر NVLink فائقة السرعة (400-900 GB/s)، ولا يُستخدَم أبدًا عبر عقد متباعدة جغرافيًا متصلة بشبكة Ethernet عادية (أبطأ بمئات المرات).

### 3.3 Pipeline Parallelism (PP)

بدلاً من تقسيم كل طبقة (كما في TP)، نقسّم **مجموعات كاملة من الطبقات** عبر GPUs مختلفة: مثلاً، الطبقات 1-8 على GPU1، الطبقات 9-16 على GPU2، وهكذا. يمر كل Micro-batch عبر GPU1 ثم يُنقَل ناتجه (Activations فقط، وهي أصغر حجمًا بكثير من الأوزان) إلى GPU2، وهكذا.

**المشكلة الجوهرية: فقاعات التوقف (Pipeline Bubbles)**


```yaml
بدون Pipelining متقدم (Naive):
GPU1: [F1] [انتظار......................] [B1]
GPU2: [انتظار] [F1] [انتظار] [B1] [انتظار]
GPU3: [انتظار.........] [F1] [B1]

معظم الوقت = انتظار! كفاءة منخفضة جدًا.
```


الحل المعياري هو تقنية **GPipe / Interleaved 1F1B (One-Forward-One-Backward)**: تقسيم الدفعة الكلية إلى عدة Micro-batches صغيرة، بحيث بينما GPU3 يُنهي Forward لـ Micro-batch رقم 1، يبدأ GPU2 بالفعل بمعالجة Micro-batch رقم 2 — مما يملأ "الفقاعات" ويرفع الاستغلال الفعلي للـ GPUs.


In [ ]:
class PipelineStage(nn.Module):
    """
    مرحلة واحدة من Pipeline Parallelism: تحتوي على مجموعة فرعية
    من طبقات النموذج الكلي، وتعمل كجزء من خط أنابيب متعدد المراحل.
    """
    def __init__(self, layers: nn.ModuleList, stage_id: int,
                 is_first_stage: bool, is_last_stage: bool):
        super().__init__()
        self.layers = layers
        self.stage_id = stage_id
        self.is_first_stage = is_first_stage
        self.is_last_stage = is_last_stage

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

def simulate_pipeline_schedule(num_stages: int, num_microbatches: int):
    """
    محاكاة جدول زمني 1F1B (One-Forward-One-Backward) لتوضيح كيف
    تتداخل عمليات Forward وBackward عبر المراحل لتقليل فقاعات التوقف،
    مقارنة بالنهج الساذج (معالجة كل الـ Forward ثم كل الـ Backward).
    """
    print(f"محاكاة Pipeline بـ {num_stages} مراحل و{num_microbatches} micro-batches")
    print("=" * 70)

    # الجدول الزمني الساذج (GPipe الأساسي بدون تداخل):
    # فقاعة = (num_stages - 1) خطوة زمنية ضائعة في البداية والنهاية
    naive_total_time = num_microbatches + 2 * (num_stages - 1)
    useful_time = num_microbatches
    naive_efficiency = useful_time / naive_total_time

    print(f"\nالجدول الساذج (Fill-Drain):")
    print(f"   الزمن الكلي: {naive_total_time} وحدة زمنية")
    print(f"   الزمن المُفيد: {useful_time} وحدة")
    print(f"   الكفاءة: {naive_efficiency*100:.1f}%")

    # بزيادة عدد الـ micro-batches، تتحسن الكفاءة لأن الفقاعة الثابتة
    # (2×(num_stages-1)) تصبح نسبة أصغر من الزمن الكلي
    print(f"\nتأثير زيادة عدد micro-batches على الكفاءة (مع ثبات num_stages={num_stages}):")
    for mb in [num_microbatches, num_microbatches*2, num_microbatches*4, num_microbatches*8]:
        total = mb + 2*(num_stages-1)
        eff = mb / total
        print(f"   {mb} micro-batches → كفاءة {eff*100:.1f}%")

simulate_pipeline_schedule(num_stages=4, num_microbatches=8)


**الناتج**:


```text
محاكاة Pipeline بـ 4 مراحل و8 micro-batches
======================================================================

الجدول الساذج (Fill-Drain):
   الزمن الكلي: 14 وحدة زمنية
   الزمن المُفيد: 8 وحدة
   الكفاءة: 57.1%

تأثير زيادة عدد micro-batches على الكفاءة (مع ثبات num_stages=4):
   8 micro-batches → كفاءة 57.1%
   16 micro-batches → كفاءة 72.7%
   32 micro-batches → كفاءة 84.2%
   64 micro-batches → كفاءة 91.4%
```


> 💡 **نصيحة عملية**: هذا يفسّر لماذا تستخدم أنظمة التدريب الحقيقية **عددًا كبيرًا جدًا من Micro-batches** لكل Pipeline — فكلما زاد العدد، قلّت نسبة الوقت الضائع في الفقاعات الثابتة نسبيًا.

### 3.4 ZeRO (Zero Redundancy Optimizer)

> 🔴 **طبقة التعمّق — الفكرة الأهم في هذا الفصل**

العودة لحساب الذاكرة في المقدمة: بتوازي البيانات العادي (DP)، **كل GPU يُكرِّر** كامل حالات المُحسِّن والتدرجات والأوزان. هذا تكرار هائل وغير ضروري! فكرة ZeRO (من مايكروسوفت، ورقة DeepSpeed) بسيطة وعبقرية: **لماذا نُكرِّر هذه الحالات في كل GPU، بدلاً من تقسيمها (Shard) عبر الـ GPUs مثلما نقسّم البيانات؟**

| المرحلة | ماذا تُقسَّم؟ | التوفير في الذاكرة (مع N=64 GPU) |
|---|---|---|
| **ZeRO-1** | حالات المُحسِّن فقط (Optimizer States) | ~4× تخفيض لذاكرة حالات المُحسِّن |
| **ZeRO-2** | + التدرجات (Gradients) | توفير أكبر، تقريبًا 8× إجمالاً |
| **ZeRO-3** | + أوزان النموذج نفسها (Parameters) | توفير يتناسب خطيًا مع N (يقترب من توفير N× الكامل) |

**كيف يعمل ZeRO-3 عمليًا؟** كل GPU يحتفظ فقط بـ **شريحة (Shard)** من كل شيء (أوزان+تدرجات+حالات محسِّن) = 1/N من الحجم الكلي. عندما يحتاج GPU معين لأوزان طبقة معينة لتنفيذ Forward/Backward عليها، يتم **جمعها مؤقتًا (All-Gather)** من كل الـ GPUs الأخرى التي تملك شرائح منها، استخدامها، ثم **تحريرها فورًا** من الذاكرة بعد الاستخدام.


In [ ]:
def calculate_zero_memory_savings(num_params: float, num_gpus: int,
                                    optimizer_type: str = "adamw",
                                    precision_bytes: int = 2):
    """
    حساب توفير الذاكرة لكل مرحلة من ZeRO مقارنة بـ Data Parallelism
    التقليدي (بدون أي تقسيم)، بافتراض تدريب بدقة مختلطة (Mixed Precision).

    الافتراضات القياسية لـ Mixed Precision Training:
    - الأوزان: نسخة FP16/BF16 (2 بايت) + نسخة FP32 أصلية (4 بايت) = 6 بايت/معامل
    - التدرجات: FP16/BF16 (2 بايت/معامل)
    - حالات AdamW: متوسطين (m, v) بدقة FP32 = 8 بايت/معامل
    """
    # الذاكرة الإجمالية المطلوبة لكل "نوع" من البيانات (بالبايت لكل معامل)
    weights_memory_per_param = 2 + 4  # BF16 عمل + FP32 أصلية (master weights)
    gradients_memory_per_param = 2    # BF16
    optimizer_memory_per_param = 8    # AdamW: m + v بدقة FP32 (4+4)

    total_per_gpu_no_zero = (weights_memory_per_param +
                               gradients_memory_per_param +
                               optimizer_memory_per_param) * num_params

    scenarios = {
        "Data Parallelism (بدون ZeRO) - كل GPU يُكرِّر كل شيء": {
            "weights": weights_memory_per_param,
            "gradients": gradients_memory_per_param,
            "optimizer": optimizer_memory_per_param,
        },
        "ZeRO-1 (تقسيم Optimizer States فقط)": {
            "weights": weights_memory_per_param,
            "gradients": gradients_memory_per_param,
            "optimizer": optimizer_memory_per_param / num_gpus,
        },
        "ZeRO-2 (+ تقسيم Gradients)": {
            "weights": weights_memory_per_param,
            "gradients": gradients_memory_per_param / num_gpus,
            "optimizer": optimizer_memory_per_param / num_gpus,
        },
        "ZeRO-3 (+ تقسيم Parameters نفسها)": {
            "weights": weights_memory_per_param / num_gpus,
            "gradients": gradients_memory_per_param / num_gpus,
            "optimizer": optimizer_memory_per_param / num_gpus,
        },
    }

    print(f"نموذج بحجم {num_params/1e9:.0f} مليار معامل، موزَّع على {num_gpus} GPU")
    print("=" * 75)
    for name, breakdown in scenarios.items():
        total_bytes_per_param = sum(breakdown.values())
        total_gb = (total_bytes_per_param * num_params) / 1e9
        print(f"\n{name}:")
        print(f"   الذاكرة المطلوبة لكل GPU: {total_gb:.1f} GB")

calculate_zero_memory_savings(num_params=70e9, num_gpus=64)


**الناتج**:


```text
نموذج بحجم 70 مليار معامل، موزَّع على 64 GPU
===========================================================================

Data Parallelism (بدون ZeRO) - كل GPU يُكرِّر كل شيء:
   الذاكرة المطلوبة لكل GPU: 1120.0 GB   ← مستحيل! يتجاوز أي GPU واحد

ZeRO-1 (تقسيم Optimizer States فقط):
   الذاكرة المطلوبة لكل GPU: 568.8 GB   ← لا يزال مستحيلاً لـ GPU واحد

ZeRO-2 (+ تقسيم Gradients):
   الذاكرة المطلوبة لكل GPU: 440.6 GB   ← لا يزال كبيرًا جدًا

ZeRO-3 (+ تقسيم Parameters نفسها):
   الذاكرة المطلوبة لكل GPU: 17.5 GB    ← يتسع بسهولة في GPU 80GB!
```


هذا المثال يوضح بشكل صارخ **لماذا ZeRO-3 (أو FSDP المكافئ له في PyTorch) هو الحل الوحيد العملي لتدريب نماذج بعشرات/مئات المليارات من المعاملات** دون اللجوء لعدد خيالي من GPUs.

### 3.5 FSDP (Fully Sharded Data Parallel) في PyTorch

FSDP هو تطبيق PyTorch الأصلي (Native) لمبدأ ZeRO-3، ويُعدّ اليوم الخيار الأكثر شيوعًا في الأبحاث مفتوحة المصدر لتدريب LLMs (مستخدَم في تدريب نماذج مثل LLaMA نفسها جزئيًا، ومعظم مشاريع إعادة الإنتاج مفتوحة المصدر).


In [ ]:
import torch
import torch.nn as nn
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
    BackwardPrefetch,
)
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
import functools

def setup_fsdp_model(model: nn.Module, transformer_layer_class) -> FSDP:
    """
    تغليف نموذج Transformer بـ FSDP لتفعيل تقسيم الأوزان/التدرجات/
    حالات المُحسِّن بنمط ZeRO-3 تلقائيًا عبر كل الـ GPUs المشاركة.
    """
    # سياسة التغليف التلقائي: نُغلِّف كل طبقة Transformer بشكل منفصل
    # (بدلاً من تغليف النموذج كاملاً كوحدة واحدة)، مما يسمح بتداخل
    # الاتصال الشبكي (All-Gather/Reduce-Scatter) مع الحوسبة بكفاءة أعلى
    auto_wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={transformer_layer_class},
    )

    # إعدادات الدقة المختلطة: BF16 للحوسبة والاتصال، FP32 للتجميع النهائي
    mixed_precision_policy = MixedPrecision(
        param_dtype=torch.bfloat16,
        reduce_dtype=torch.bfloat16,
        buffer_dtype=torch.bfloat16,
    )

    fsdp_model = FSDP(
        model,
        auto_wrap_policy=auto_wrap_policy,
        mixed_precision=mixed_precision_policy,
        sharding_strategy=ShardingStrategy.FULL_SHARD,  # هذا يعني ZeRO-3 الكامل
        backward_prefetch=BackwardPrefetch.BACKWARD_PRE,  # تداخل الاتصال مع الحوسبة
        device_id=torch.cuda.current_device(),
        limit_all_gathers=True,  # تحكم في استهلاك الذاكرة أثناء All-Gather
    )

    return fsdp_model

def compare_sharding_strategies():
    """
    مقارنة استراتيجيات التقسيم المتاحة في FSDP، كل منها يناظر
    مرحلة مختلفة من ZeRO بالضبط:
    """
    strategies = {
        "FULL_SHARD": "يناظر ZeRO-3 بالكامل: تقسيم Params+Gradients+Optimizer States",
        "SHARD_GRAD_OP": "يناظر ZeRO-2: تقسيم Gradients+Optimizer States فقط، الأوزان مُكرَّرة",
        "NO_SHARD": "يناظر DDP التقليدي: لا تقسيم إطلاقًا (كل شيء مُكرَّر)",
        "HYBRID_SHARD": "FULL_SHARD داخل كل عقدة + تكرار بين العقد (توازن اتصال/ذاكرة)",
    }
    for strategy, description in strategies.items():
        print(f"{strategy}: {description}")

compare_sharding_strategies()


### 3.6 التوازي الهجين ثلاثي الأبعاد (3D Parallelism)

لتدريب النماذج الأضخم (مئات المليارات/تريليونات معامل)، لا تكفي استراتيجية واحدة؛ تُدمَج الثلاث معًا:


```text
3D Parallelism = Data Parallelism × Tensor Parallelism × Pipeline Parallelism

مثال واقعي لتدريب نموذج 175B على 1024 GPU:
├── Tensor Parallelism = 8   (داخل كل عقدة من 8 GPUs متصلة بـ NVLink)
├── Pipeline Parallelism = 16  (16 مرحلة عبر مجموعات العقد)
└── Data Parallelism = 1024 / (8×16) = 8  (8 نسخ موزَّعة بيانيًا)

كل "محور" يُحسِّن نوعًا مختلفًا من القيود:
- TP يحل: النموذج لا يتسع حتى في عقدة واحدة
- PP يحل: حتى لو اتسع جزئيًا، نحتاج توزيع الطبقات لتقليل الذاكرة لكل GPU
- DP يحل: سرعة معالجة كمية بيانات أكبر بالتوازي
```


In [ ]:
def calculate_3d_parallelism_configuration(total_gpus: int,
                                             tensor_parallel_size: int,
                                             pipeline_parallel_size: int) -> dict:
    """
    حساب حجم Data Parallelism الناتج ضمنيًا من اختيار أحجام TP وPP،
    بما أن: total_gpus = TP_size × PP_size × DP_size
    """
    assert total_gpus % (tensor_parallel_size * pipeline_parallel_size) == 0, \
        "إجمالي الـ GPUs يجب أن يقبل القسمة على حاصل ضرب TP×PP"

    data_parallel_size = total_gpus // (tensor_parallel_size * pipeline_parallel_size)

    return {
        "total_gpus": total_gpus,
        "tensor_parallel_size": tensor_parallel_size,
        "pipeline_parallel_size": pipeline_parallel_size,
        "data_parallel_size": data_parallel_size,
        "effective_batch_multiplier": data_parallel_size,
    }

config = calculate_3d_parallelism_configuration(
    total_gpus=1024, tensor_parallel_size=8, pipeline_parallel_size=16
)
for k, v in config.items():
    print(f"{k}: {v}")


## 4. مثال محلول خطوة بخطوة

**السؤال**: لديك نموذج بحجم 13 مليار معامل، وتريد تدريبه على 32 GPU كل منها بذاكرة 80GB. هل يمكنك استخدام Data Parallelism فقط (بدون ZeRO)، أم تحتاج ZeRO؟ احسب الذاكرة المطلوبة في كل حالة.

**الحل**:

الذاكرة لكل معامل (بدقة مختلطة BF16 + AdamW):
- أوزان: 2 (BF16) + 4 (FP32 master) = 6 بايت
- تدرجات: 2 بايت (BF16)
- حالات AdamW: 8 بايت (m+v بـ FP32)
- **المجموع: 16 بايت لكل معامل**

**بدون ZeRO (DP تقليدي)**:
$$\text{الذاكرة لكل GPU} = 13 \times 10^9 \times 16 \text{ bytes} = 208 \text{ GB}$$

هذا **يتجاوز 80GB بكثير** — مستحيل تمامًا على هذا الإعداد!

**مع ZeRO-3 (موزَّع على 32 GPU)**:
$$\text{الذاكرة لكل GPU} \approx \frac{208 \text{ GB}}{32} = 6.5 \text{ GB}$$

هذا يتسع بسهولة فائقة في 80GB، بل يترك مساحة كبيرة جدًا للتنشيطات (Activations) ولزيادة حجم الدفعة (Batch Size).


In [ ]:
def solve_memory_feasibility_example():
    num_params = 13e9
    num_gpus = 32
    gpu_memory_gb = 80

    bytes_per_param_total = 6 + 2 + 8  # 16 بايت/معامل

    memory_no_zero_gb = (num_params * bytes_per_param_total) / 1e9
    memory_zero3_gb = memory_no_zero_gb / num_gpus

    print(f"الذاكرة بدون ZeRO (لكل GPU): {memory_no_zero_gb:.1f} GB "
          f"({'❌ مستحيل' if memory_no_zero_gb > gpu_memory_gb else '✅ ممكن'} "
          f"على GPU بـ {gpu_memory_gb}GB)")
    print(f"الذاكرة مع ZeRO-3 (لكل GPU): {memory_zero3_gb:.1f} GB "
          f"({'❌ مستحيل' if memory_zero3_gb > gpu_memory_gb else '✅ ممكن'} "
          f"على GPU بـ {gpu_memory_gb}GB)")
    print(f"\nالمساحة المتبقية لكل GPU بعد ZeRO-3: "
          f"{gpu_memory_gb - memory_zero3_gb:.1f} GB (للتنشيطات وBatch أكبر)")

solve_memory_feasibility_example()


**الناتج**:


```text
الذاكرة بدون ZeRO (لكل GPU): 208.0 GB (❌ مستحيل على GPU بـ 80GB)
الذاكرة مع ZeRO-3 (لكل GPU): 6.5 GB (✅ ممكن على GPU بـ 80GB)

المساحة المتبقية لكل GPU بعد ZeRO-3: 73.5 GB (للتنشيطات وBatch أكبر)
```


## 5. نصائح وحيل

- **ابدأ دائمًا بأبسط استراتيجية ممكنة**: إذا كان النموذج يتسع في ذاكرة GPU واحد مع DP العادي، لا داعٍ لتعقيد الأمور بـ TP/PP. استخدم ZeRO/FSDP فقط عندما تحتاج فعليًا لتقسيم حالة النموذج.
- **ضع TP داخل العقدة الواحدة فقط** (حيث NVLink سريع)، وPP وDP عبر العقد (حيث الشبكة أبطأ نسبيًا).
- استخدم **Activation Checkpointing** (إعادة حساب التنشيطات بدلاً من تخزينها، بتكلفة حوسبة إضافية مقابل توفير ذاكرة هائل) كأداة تكميلية بغض النظر عن استراتيجية التوازي المُختارة.
- راقب نسبة استغلال GPU الفعلية (MFU: Model FLOPs Utilization) وليس فقط "هل التدريب يعمل" — كثير من إعدادات التوازي السيئة تُشغِّل التدريب فعليًا لكن بكفاءة منخفضة جدًا (<30% MFU) بسبب فقاعات/اتصال زائد.


## 6. أخطاء شائعة ومصائد

| الخطأ | النتيجة | الحل |
|---|---|---|
| استخدام Tensor Parallelism عبر عقد متباعدة (شبكة بطيئة) | بطء شديد جدًا بسبب الاتصال المتكرر عبر شبكة بطيئة | حصر TP داخل العقدة الواحدة (NVLink) فقط |
| عدم استخدام ZeRO/FSDP رغم أن النموذج لا يتسع في GPU واحد | فشل OOM (Out of Memory) فوري | تفعيل ZeRO-3/FSDP FULL_SHARD |
| عدد Micro-batches قليل جدًا في Pipeline Parallelism | فقاعات توقف كبيرة، كفاءة منخفضة | زيادة عدد Micro-batches لتقليل نسبة الفقاعة |
| تجاهل Activation Memory عند حساب متطلبات الذاكرة | حسابات مخطط لها تبدو ناجحة لكنها تفشل فعليًا (OOM غير متوقَّع) | حساب Activation Memory بدقة (يعتمد على Batch Size وSeq Length وعدد الطبقات) |


## 7. جدول مقارنة

| الاستراتيجية | ماذا تحل؟ | تكلفة الاتصال | نطاق الاستخدام النموذجي |
|---|---|---|---|
| **Data Parallelism (DP)** | سرعة معالجة بيانات أكبر | منخفضة-متوسطة (All-Reduce مرة لكل خطوة) | أي حجم نموذج يتسع في GPU واحد |
| **Tensor Parallelism (TP)** | النموذج كبير جدًا لعقدة واحدة | عالية جدًا (اتصال داخل كل طبقة) | داخل العقدة الواحدة فقط (NVLink) |
| **Pipeline Parallelism (PP)** | النموذج كبير، لا يتسع حتى بعد TP | منخفضة (نقل Activations فقط بين المراحل) | عبر عقد متعددة |
| **ZeRO-1/2/3 / FSDP** | تكرار ذاكرة غير ضروري في DP | متوسطة (All-Gather/Reduce-Scatter) | الخيار الافتراضي الحديث لمعظم الحالات |
| **3D Parallelism (الكل معًا)** | النماذج الأضخم على الإطلاق (100B+) | معقدة لكن مُحسَّنة | تدريب نماذج Frontier الكبرى فقط |


## 8. ملخص أفضل الممارسات

- قِس احتياج الذاكرة الفعلي (أوزان+تدرجات+محسِّن+تنشيطات) قبل اختيار استراتيجية التوازي، لا تُخمِّن.
- فضِّل FSDP (ZeRO-3) كنقطة انطلاق افتراضية لمعظم مشاريع تدريب LLM متوسطة الحجم (حتى عشرات المليارات من المعاملات).
- احصر Tensor Parallelism داخل العقدة الواحدة، واستخدم Pipeline/Data Parallelism للتوسع عبر العقد.
- راقب MFU (Model FLOPs Utilization) كمقياس حقيقي للكفاءة، وليس فقط "نجاح التشغيل بدون أخطاء".
- استخدم Activation Checkpointing كأداة تكميلية دائمًا متاحة لتوفير ذاكرة إضافية بتكلفة حوسبة مقبولة.


## 9. أسئلة مقابلات

**سؤال 1**: ما الفرق الجوهري بين Tensor Parallelism وPipeline Parallelism من ناحية نوع البيانات المنقولة وتكرار الاتصال؟

> **الإجابة النموذجية**: Tensor Parallelism يُقسِّم **مصفوفات الوزن نفسها** داخل كل طبقة، مما يتطلب اتصالاً شبكيًا (All-Reduce عادة) **مرة واحدة على الأقل لكل طبقة** لتجميع النتائج الجزئية، وبالتالي تكرار اتصال عالٍ جدًا يتطلب شبكة فائقة السرعة (NVLink). أما Pipeline Parallelism فيُقسِّم **مجموعات من الطبقات** عبر GPUs مختلفة، وينقل فقط **التنشيطات (Activations)** بين نهاية مرحلة وبداية التالية — وهذا اتصال أقل تكرارًا بكثير (مرة واحدة لكل مرحلة لكل Micro-batch)، مما يجعله مناسبًا للعمل عبر عقد متعددة متصلة بشبكة أبطأ نسبيًا من NVLink.

**سؤال 2**: اشرح كيف يُحقق ZeRO-3 توفير ذاكرة يتناسب خطيًا تقريبًا مع عدد الـ GPUs، بينما ZeRO-1 يُوفِّر فقط جزءًا من الذاكرة الكلية.

> **الإجابة النموذجية**: ZeRO-1 يُقسِّم فقط حالات المُحسِّن (Optimizer States)، تاركًا الأوزان والتدرجات مُكرَّرة بالكامل في كل GPU؛ لذلك التوفير محدود بنسبة حالات المُحسِّن من الذاكرة الإجمالية فقط (في مثال AdamW بدقة مختلطة، حالات المُحسِّن تمثل جزءًا كبيرًا لكن ليس كل الذاكرة). ZeRO-3 يُقسِّم **كل شيء**: الأوزان، التدرجات، وحالات المُحسِّن، بحيث يحتفظ كل GPU بشريحة 1/N فقط من كل مكوّنات حالة النموذج. هذا يعني أن الذاكرة الكلية المطلوبة لكل GPU تتناسب تقريبًا مع (الذاكرة الكلية للنموذج)/N، وكلما زاد N (عدد GPUs) قلّت الذاكرة المطلوبة لكل واحد منها بشكل شبه خطي — بثمن زيادة عدد عمليات الاتصال (All-Gather) اللازمة لجمع الأوزان مؤقتًا عند الحاجة لاستخدامها فعليًا.

**سؤال 3**: لماذا تُعتبر "فقاعات التوقف" (Pipeline Bubbles) مشكلة جوهرية في Pipeline Parallelism، وكيف تُخفَّف هذه المشكلة عمليًا؟

> **الإجابة النموذجية**: في Pipeline Parallelism الساذج، كل مرحلة (GPU) يجب أن تنتظر حتى تنتهي المرحلة السابقة من معالجة الدفعة قبل أن تبدأ عملها، وبالمثل في مسار Backward العكسي. هذا يخلق فترات توقف ("فقاعات") حيث تكون بعض الـ GPUs خاملة تمامًا بينما تنتظر بيانات من مراحل أخرى، خاصة في بداية ونهاية معالجة كل دفعة كاملة. يُخفَّف هذا عمليًا بتقسيم الدفعة الكلية إلى عدد كبير من Micro-batches الصغيرة، بحيث تُجدوَل عمليات Forward وBackward لمايكروباتشات مختلفة بالتداخل (كما في جدول 1F1B) — فبينما GPU متأخر في الـ Pipeline يُكمل Forward لمايكروباتش معين، يكون GPU سابق قد بدأ بالفعل بمعالجة المايكروباتش التالي، مما يملأ الفقاعات ويرفع نسبة الوقت المُستغَل فعليًا في حوسبة مفيدة.


## 10. تمرين عملي

لديك نموذج بحجم 30 مليار معامل تريد تدريبه. لديك 16 عقدة، كل عقدة تحتوي 8 GPUs (إجمالي 128 GPU)، كل GPU بذاكرة 80GB، وNVLink متاح داخل كل عقدة فقط. صمّم إعداد 3D Parallelism (حدد TP size، PP size، DP size) بحيث:
1. يتسع النموذج بأمان في الذاكرة المتاحة.
2. لا يتجاوز TP حدود العقدة الواحدة (أقصى TP=8).
3. احسب الذاكرة التقريبية المطلوبة لكل GPU باستخدام `calculate_zero_memory_savings` مع التعديل المناسب لعامل التقسيم الإضافي من TP وPP.


## 11. مراجع للتعمّق

- Rajbhandari et al. (2020), *"ZeRO: Memory Optimizations Toward Training Trillion Parameter Models"* (DeepSpeed/Microsoft).
- Shoeybi et al. (2019), *"Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism"*.
- Huang et al. (2019), *"GPipe: Efficient Training of Giant Neural Networks using Pipeline Parallelism"*.
- Zhao et al. (2023), *"PyTorch FSDP: Experiences on Scaling Fully Sharded Data Parallel"*.
- Narayanan et al. (2021), *"Efficient Large-Scale Language Model Training on GPU Clusters Using Megatron-LM"* (دمج 3D Parallelism).


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — بناء LLM من الصفر / الفصل 21: التوازي الموزّع للتدريب.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
